# Wine Quality Project

This notebook is a portfolio-style working draft for the project:

1. Wine quality deep learning + analytical agent.


## Project 1 - Wine quality DL + agentic analytical workflow

This section loads `wine_quality.csv`, trains a binary high-quality classifier, exports metrics and feature importance, and defines a small analytical agent that can answer multi-step questions about the strongest chemical predictors of high-quality wine.

In [1]:
# Project 1 - Environment & imports
import os
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score, confusion_matrix, classification_report
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

try:
    import shap
except Exception:
    shap = None

sns.set_theme(style="whitegrid")
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Project 1 - Load wine_quality.csv
wine_path = Path("wine_quality.csv")
wine_df = pd.read_csv(wine_path)
wine_df.columns = [c.strip().replace(" ", "_") for c in wine_df.columns]
wine_df["high_quality"] = (wine_df["quality"] >= 7).astype(int)
clean_wine_path = Path("wine_agent") / "clean_wine_quality.csv"
clean_wine_path.parent.mkdir(parents=True, exist_ok=True)
wine_df.to_csv(clean_wine_path, index=False)
print(wine_df.head(3))
print(wine_df.dtypes)

# Project 1 - EDA & preprocessing
summary = wine_df.describe(include="all").T
summary.to_csv(Path("wine_agent") / "wine_summary_stats.csv")
plt.figure(figsize=(10, 6))
sns.histplot(wine_df["quality"], bins=10, kde=True)
plt.title("Wine quality distribution")
plt.tight_layout()
plt.savefig(Path("wine_agent") / "wine_quality_distribution.png", dpi=160)
plt.close()

corr = wine_df.drop(columns=["type"]).corr(numeric_only=True)
plt.figure(figsize=(12, 8))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Wine correlations")
plt.tight_layout()
plt.savefig(Path("wine_agent") / "wine_correlation_heatmap.png", dpi=160)
plt.close()

# Project 1 - Feature engineering & train/val split
wine_model_df = pd.get_dummies(wine_df.drop(columns=["quality"]), columns=["type"], drop_first=True)
feature_names = wine_model_df.drop(columns=["high_quality"]).columns.tolist()
X = wine_model_df.drop(columns=["high_quality"]).astype(np.float32).values
y = wine_model_df["high_quality"].astype(np.float32).values
x_train, x_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y)
x_val, x_test, y_val, y_test = train_test_split(x_temp, y_temp, test_size=0.5, random_state=RANDOM_SEED, stratify=y_temp)
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_val = scaler.transform(x_val)
x_test = scaler.transform(x_test)
pd.DataFrame(x_train, columns=feature_names).to_csv(Path("wine_agent") / "wine_train_split.csv", index=False)
pd.DataFrame(x_val, columns=feature_names).to_csv(Path("wine_agent") / "wine_val_split.csv", index=False)
pd.DataFrame(x_test, columns=feature_names).to_csv(Path("wine_agent") / "wine_test_split.csv", index=False)

# Project 1 - Train DL model
class WineNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

model = WineNet(x_train.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()
train_loader = DataLoader(TensorDataset(torch.tensor(x_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32)), batch_size=64, shuffle=True)

history = []
best_state = None
best_val_auc = -1
for epoch in range(1, 11):
    model.train()
    epoch_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * batch_x.size(0)
    model.eval()
    with torch.no_grad():
        val_probs = torch.sigmoid(model(torch.tensor(x_val, dtype=torch.float32).to(device))).cpu().numpy()
    val_pred = (val_probs >= 0.5).astype(int)
    val_auc = roc_auc_score(y_val, val_probs)
    val_acc = accuracy_score(y_val, val_pred)
    history.append({"epoch": epoch, "train_loss": epoch_loss / len(train_loader.dataset), "val_auc": val_auc, "val_accuracy": val_acc})
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_state = {"model_state_dict": model.state_dict(), "scaler_mean": scaler.mean_, "scaler_scale": scaler.scale_, "feature_names": feature_names}

pd.DataFrame(history).to_csv(Path("wine_agent") / "wine_training_metrics.csv", index=False)
torch.save(best_state, Path("wine_agent") / "wine_classifier.pth")

# Project 1 - Model evaluation & CSV outputs
model.load_state_dict(best_state["model_state_dict"])
model.eval()
with torch.no_grad():
    test_probs = torch.sigmoid(model(torch.tensor(x_test, dtype=torch.float32).to(device))).cpu().numpy()
test_pred = (test_probs >= 0.5).astype(int)
metrics = {
    "accuracy": accuracy_score(y_test, test_pred),
    "roc_auc": roc_auc_score(y_test, test_probs),
    "average_precision": average_precision_score(y_test, test_probs),
}
pd.DataFrame([metrics]).to_csv(Path("wine_agent") / "wine_test_metrics.csv", index=False)
pd.DataFrame({"actual": y_test.astype(int), "predicted_probability": test_probs, "predicted_label": test_pred}).to_csv(Path("wine_agent") / "wine_test_predictions.csv", index=False)
print(metrics)

# Project 1 - Explainability (feature importance / SHAP)
perm_importance = []
rng = np.random.default_rng(RANDOM_SEED)
base_acc = metrics["accuracy"]
for i, feature in enumerate(feature_names):
    permuted = x_val.copy()
    rng.shuffle(permuted[:, i])
    with torch.no_grad():
        perm_probs = torch.sigmoid(model(torch.tensor(permuted, dtype=torch.float32).to(device))).cpu().numpy()
    perm_acc = accuracy_score(y_val, (perm_probs >= 0.5).astype(int))
    perm_importance.append({"feature": feature, "baseline_accuracy": base_acc, "permuted_accuracy": perm_acc, "importance_drop": base_acc - perm_acc})
importance_df = pd.DataFrame(perm_importance).sort_values("importance_drop", ascending=False)
importance_df.to_csv(Path("wine_agent") / "wine_feature_importance.csv", index=False)
plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df.head(10), x="importance_drop", y="feature")
plt.title("Top wine predictors")
plt.tight_layout()
plt.savefig(Path("wine_agent") / "wine_feature_importance.png", dpi=160)
plt.close()

# Project 1 - Agentic analytical workflow
class WineAnalystAgent:
    def __init__(self, metrics_df, importance_df):
        self.metrics_df = metrics_df
        self.importance_df = importance_df
    def answer(self, question: str) -> str:
        q = question.lower()
        best = self.metrics_df.sort_values("val_auc", ascending=False).iloc[0]
        top = self.importance_df.head(5)
        if "high-quality" in q or "predict" in q:
            lines = [f"- {row.feature}: importance drop {row.importance_drop:.4f}" for row in top.itertuples(index=False)]
            return f"Best validation AUC: {best.val_auc:.3f}, accuracy: {best.val_accuracy:.3f}. Top predictors:\n" + "\n".join(lines)
        return "Ask about model performance or which chemical factors are strongest."

agent = WineAnalystAgent(pd.DataFrame(history), importance_df)
print(agent.answer("What chemical factors most strongly predict high-quality wine?"))

# Project 1 - Export artifacts
manifest = {
    "model": "wine_agent/wine_classifier.pth",
    "metrics": ["wine_agent/wine_training_metrics.csv", "wine_agent/wine_test_metrics.csv"],
    "predictions": "wine_agent/wine_test_predictions.csv",
    "importance": "wine_agent/wine_feature_importance.csv",
}
(Path("wine_agent") / "wine_reproducibility_manifest.json").write_text(json.dumps(manifest, indent=2))
print(manifest)


Using device: cpu
    type  fixed_acidity  volatile_acidity  citric_acid  residual_sugar  \
0  white            7.0              0.27         0.36            20.7   
1  white            6.3              0.30         0.34             1.6   
2  white            8.1              0.28         0.40             6.9   

   chlorides  free_sulfur_dioxide  total_sulfur_dioxide  density    pH  \
0      0.045                 45.0                 170.0   1.0010  3.00   
1      0.049                 14.0                 132.0   0.9940  3.30   
2      0.050                 30.0                  97.0   0.9951  3.26   

   sulphates  alcohol  quality  high_quality  
0       0.45      8.8        6             0  
1       0.49      9.5        6             0  
2       0.44     10.1        6             0  
type                     object
fixed_acidity           float64
volatile_acidity        float64
citric_acid             float64
residual_sugar          float64
chlorides               float64
free_sul

ValueError: Input contains NaN.